In [63]:
import numpy as np
import pandas as pd
import time
import itertools
import contextlib
import io
from pathlib import Path

from raiselab.core import dvqe

import gurobipy as gp
from gurobipy import GRB
GUROBI_AVAILABLE = True



# ============================================================
# EXPERIMENT SETTINGS
# ============================================================

N_VALUES = list(range(3, 19))          # n = 3, 4, ..., 18
SEEDS = [1, 2, 3, 4, 5]               # five random QUBO instances per size

OUTPUT_DIR = Path("dvqe_dbh_vs_gurobi_results")
OUTPUT_DIR.mkdir(exist_ok=True)

# Stronger DVQE settings than the ablation study.
# The goal here is higher D-BH reliability, not equal low-budget comparison.
DBH_SETTINGS = dict(
    depth=1,
    lr=0.5,
    max_iters=80,
    rel_tol=1e-4,
    num_shots=512,
    final_shots=5000,
    warm_start_population=12,
    warm_start_iters=20,
    warm_start_shots=512,
    energy_mode="cvar",
    cvar_alpha=0.1,
    verbose=False,
)

# D-BH configuration
DBH_METHOD = dict(
    method="D-BH",
    mode="distributed",
    init_type=2,
    qpu_qubit_config=[4, 4, 4, 4, 4],   # total capacity = 20 qubits, enough for n <= 18
)

# Numerical tolerances
EXACT_SUCCESS_TOL = 1e-8
NEAR_SUCCESS_TOL = 1e-4

# Gurobi settings
GUROBI_TIME_LIMIT = None       # set to a number of seconds if needed, e.g., 300
GUROBI_THREADS = 1             # use 1 for more stable timing comparisons
GUROBI_OUTPUT = 0              # 0 = silent, 1 = show Gurobi output


# ============================================================
# QUBO HELPERS
# ============================================================

def qubo_energy(Q, q, z):
    """
    Compute QUBO energy:
        C(z) = z^T Q z + q^T z
    """
    z = np.asarray(z, dtype=float)
    return float(z.T @ Q @ z + q.T @ z)


def scale_qubo(Q, q):
    """
    Scale QUBO coefficients for DVQE numerical stability.
    The minimizer is unchanged because the objective is divided
    by a positive constant.
    """
    scale = max(
        float(np.max(np.abs(Q))) if Q.size > 0 else 0.0,
        float(np.max(np.abs(q))) if q.size > 0 else 0.0,
        1.0,
    )
    return Q / scale, q / scale, scale


def hamming_distance(z1, z2):
    z1 = np.asarray(z1, dtype=int)
    z2 = np.asarray(z2, dtype=int)
    return int(np.sum(z1 != z2))


def generate_hard_random_qubo(n, seed, density=0.85, coupling_scale=1.0, linear_scale=0.35):
    """
    Generate a harder random QUBO instance.

    The generator creates a dense, frustrated QUBO with mixed-sign pairwise
    interactions. This is harder than a sparse or mostly diagonal QUBO because
    many variables interact and local bit flips can have competing effects.

    Parameters
    ----------
    n : int
        Number of binary variables.
    seed : int
        Random seed.
    density : float
        Probability of keeping an off-diagonal interaction.
    coupling_scale : float
        Standard deviation of off-diagonal couplings.
    linear_scale : float
        Standard deviation of linear terms.

    Returns
    -------
    Q : ndarray
        Symmetric QUBO matrix.
    q : ndarray
        Linear QUBO vector.
    """
    rng = np.random.default_rng(seed)

    # Dense mixed-sign off-diagonal couplings
    W = rng.normal(loc=0.0, scale=coupling_scale, size=(n, n))
    mask = rng.random((n, n)) < density
    mask = np.triu(mask, 1)

    W = np.triu(W, 1) * mask
    Q = W + W.T

    # Add diagonal bias terms
    diag = rng.normal(loc=0.0, scale=0.50, size=n)
    Q[np.diag_indices(n)] = diag

    # Add smaller linear terms so pairwise frustration remains important
    q = rng.normal(loc=0.0, scale=linear_scale, size=n)

    # Add a weak random shift on selected variables to reduce excessive ties
    tie_break = rng.normal(loc=0.0, scale=0.05, size=n)
    q = q + tie_break

    # Symmetrize for numerical consistency
    Q = 0.5 * (Q + Q.T)

    return Q, q


# ============================================================
# GUROBI MIQP SOLVER
# ============================================================

def solve_qubo_gurobi_miqp(Q, q, time_limit=None, threads=1, output_flag=0):
    """
    Solve the QUBO exactly using Gurobi MIQP.

    Problem:
        min z^T Q z + q^T z
        s.t. z_i in {0,1}

    Returns
    -------
    dict with:
        z_best
        objective
        runtime_sec
        status
        mip_gap
        node_count
    """
    if not GUROBI_AVAILABLE:
        raise RuntimeError("gurobipy is not available. Please install Gurobi/gurobipy first.")

    n = q.shape[0]

    start = time.time()

    model = gp.Model("qubo_miqp")
    model.Params.OutputFlag = output_flag
    model.Params.Threads = threads

    if time_limit is not None:
        model.Params.TimeLimit = time_limit

    # Needed for general nonconvex quadratic objectives.
    # Binary QUBO is still solved globally by MIQP branch-and-bound.
    model.Params.NonConvex = 2
    model.Params.MIPGap = 0.0

    z = model.addVars(n, vtype=GRB.BINARY, name="z")

    obj = gp.QuadExpr()

    # Quadratic term z^T Q z
    for i in range(n):
        for j in range(n):
            if abs(Q[i, j]) > 0.0:
                obj += float(Q[i, j]) * z[i] * z[j]

    # Linear term q^T z
    for i in range(n):
        if abs(q[i]) > 0.0:
            obj += float(q[i]) * z[i]

    model.setObjective(obj, GRB.MINIMIZE)
    model.optimize()

    runtime = time.time() - start

    status_code = model.Status
    status_name = {
        GRB.OPTIMAL: "optimal",
        GRB.TIME_LIMIT: "time_limit",
        GRB.INFEASIBLE: "infeasible",
        GRB.INF_OR_UNBD: "inf_or_unbd",
        GRB.UNBOUNDED: "unbounded",
    }.get(status_code, f"status_{status_code}")

    if model.SolCount == 0:
        return {
            "z_best": None,
            "objective": np.nan,
            "runtime_sec": runtime,
            "status": status_name,
            "mip_gap": np.nan,
            "node_count": np.nan,
        }

    z_best = np.array([int(round(z[i].X)) for i in range(n)], dtype=int)
    objective = qubo_energy(Q, q, z_best)

    try:
        mip_gap = float(model.MIPGap)
    except Exception:
        mip_gap = np.nan

    try:
        node_count = float(model.NodeCount)
    except Exception:
        node_count = np.nan

    return {
        "z_best": z_best,
        "objective": objective,
        "runtime_sec": runtime,
        "status": status_name,
        "mip_gap": mip_gap,
        "node_count": node_count,
    }


# ============================================================
# D-BH DVQE SOLVER
# ============================================================

def solve_qubo_dbh(Q, q, n, seed):
    """
    Solve one QUBO using distributed DVQE with Black Hole warm start.
    """
    Q_train, q_train, qubo_scale = scale_qubo(Q, q)

    run_seed = 100000 + 1000 * n + seed

    start = time.time()

    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        z_best, final_circuit, hist = dvqe(
            mode=DBH_METHOD["mode"],
            Q=Q_train,
            q_linear=q_train,
            init_type=DBH_METHOD["init_type"],
            qpu_qubit_config=DBH_METHOD["qpu_qubit_config"],
            seed=run_seed,
            **DBH_SETTINGS,
        )

    runtime = time.time() - start

    z_best = np.asarray(z_best, dtype=int)
    objective = qubo_energy(Q, q, z_best)

    return {
        "z_best": z_best,
        "objective": objective,
        "runtime_sec": runtime,
        "hist_size": len(hist) if hist is not None else np.nan,
        "qubo_scale": qubo_scale,
        "status": "ok",
    }


# ============================================================
# MAIN EXPERIMENT RUNNER
# ============================================================

def run_dbh_vs_gurobi_experiment():
    """
    Compare Gurobi MIQP and D-BH DVQE on hard random QUBO instances.

    Gurobi MIQP is used as the exact classical reference solver.
    D-BH is evaluated by objective gap, success rate, Hamming distance,
    and runtime.

    The purpose is not to claim that simulator-based DVQE is faster than
    Gurobi. The purpose is to measure how reliably D-BH recovers the exact
    or near-exact QUBO optimum under stronger DVQE settings.
    """

    if not GUROBI_AVAILABLE:
        raise RuntimeError("Gurobi is not available. Please install gurobipy and activate a Gurobi license.")

    detailed_rows = []

    total_instances = len(N_VALUES) * len(SEEDS)

    print("=" * 100)
    print("D-BH DVQE VS GUROBI MIQP ON HARD RANDOM QUBO INSTANCES")
    print("=" * 100)
    print(f"Problem sizes: {N_VALUES}")
    print(f"Seeds: {SEEDS}")
    print(f"Total QUBO instances: {total_instances}")
    print(f"D-BH QPU config: {DBH_METHOD['qpu_qubit_config']}")
    print(f"D-BH settings: {DBH_SETTINGS}")
    print("=" * 100)

    for n in N_VALUES:
        for seed in SEEDS:

            qubo_seed = 1000 * n + seed
            Q, q = generate_hard_random_qubo(n=n, seed=qubo_seed)

            print(f"\nQUBO instance | n={n:2d} | seed={seed:2d} | qubo_seed={qubo_seed}")

            # ------------------------------------------------------------
            # Solve with Gurobi MIQP
            # ------------------------------------------------------------
            try:
                gurobi_result = solve_qubo_gurobi_miqp(
                    Q=Q,
                    q=q,
                    time_limit=GUROBI_TIME_LIMIT,
                    threads=GUROBI_THREADS,
                    output_flag=GUROBI_OUTPUT,
                )

                z_gurobi = gurobi_result["z_best"]
                f_gurobi = gurobi_result["objective"]

                print(
                    f"  Gurobi MIQP | "
                    f"obj={f_gurobi: .8f} | "
                    f"status={gurobi_result['status']} | "
                    f"time={gurobi_result['runtime_sec']: .4f}s"
                )

            except Exception as e:
                z_gurobi = None
                f_gurobi = np.nan
                gurobi_result = {
                    "z_best": None,
                    "objective": np.nan,
                    "runtime_sec": np.nan,
                    "status": f"failed: {str(e)}",
                    "mip_gap": np.nan,
                    "node_count": np.nan,
                }

                print(f"  Gurobi MIQP | FAILED | {str(e)}")

            # ------------------------------------------------------------
            # Solve with D-BH DVQE
            # ------------------------------------------------------------
            try:
                dbh_result = solve_qubo_dbh(
                    Q=Q,
                    q=q,
                    n=n,
                    seed=seed,
                )

                z_dbh = dbh_result["z_best"]
                f_dbh = dbh_result["objective"]

                if z_gurobi is not None and np.isfinite(f_gurobi):
                    raw_gap = float(f_dbh - f_gurobi)
                    abs_gap = max(0.0, raw_gap)
                    rel_gap = float(abs_gap / max(1.0, abs(f_gurobi)))
                    exact_success = bool(abs_gap <= EXACT_SUCCESS_TOL)
                    near_success = bool(abs_gap <= NEAR_SUCCESS_TOL)
                    ham = hamming_distance(z_dbh, z_gurobi)
                    same_solution = bool("".join(map(str, z_dbh.tolist())) == "".join(map(str, z_gurobi.tolist())))
                else:
                    raw_gap = np.nan
                    abs_gap = np.nan
                    rel_gap = np.nan
                    exact_success = False
                    near_success = False
                    ham = np.nan
                    same_solution = False

                print(
                    f"  D-BH DVQE    | "
                    f"obj={f_dbh: .8f} | "
                    f"gap={abs_gap: .3e} | "
                    f"exact={exact_success} | "
                    f"near={near_success} | "
                    f"time={dbh_result['runtime_sec']: .4f}s"
                )

            except Exception as e:
                z_dbh = None
                f_dbh = np.nan
                dbh_result = {
                    "z_best": None,
                    "objective": np.nan,
                    "runtime_sec": np.nan,
                    "hist_size": np.nan,
                    "qubo_scale": np.nan,
                    "status": f"failed: {str(e)}",
                }

                raw_gap = np.nan
                abs_gap = np.nan
                rel_gap = np.nan
                exact_success = False
                near_success = False
                ham = np.nan
                same_solution = False

                print(f"  D-BH DVQE    | FAILED | {str(e)}")

            # ------------------------------------------------------------
            # Store row
            # ------------------------------------------------------------
            row = {
                "n": n,
                "seed": seed,
                "qubo_seed": qubo_seed,

                "gurobi_objective": f_gurobi,
                "gurobi_runtime_sec": gurobi_result["runtime_sec"],
                "gurobi_status": gurobi_result["status"],
                "gurobi_mip_gap": gurobi_result["mip_gap"],
                "gurobi_node_count": gurobi_result["node_count"],
                "gurobi_z": "" if z_gurobi is None else "".join(map(str, z_gurobi.tolist())),

                "dbh_objective": f_dbh,
                "dbh_runtime_sec": dbh_result["runtime_sec"],
                "dbh_status": dbh_result["status"],
                "dbh_hist_size": dbh_result["hist_size"],
                "dbh_qubo_scale": dbh_result["qubo_scale"],
                "dbh_z": "" if z_dbh is None else "".join(map(str, z_dbh.tolist())),

                "raw_gap_dbh_minus_gurobi": raw_gap,
                "abs_gap": abs_gap,
                "rel_gap": rel_gap,
                "exact_success": exact_success,
                "near_success": near_success,
                "hamming_to_gurobi": ham,
                "same_solution_string": same_solution,
                "runtime_ratio_dbh_over_gurobi": (
                    dbh_result["runtime_sec"] / gurobi_result["runtime_sec"]
                    if np.isfinite(dbh_result["runtime_sec"])
                    and np.isfinite(gurobi_result["runtime_sec"])
                    and gurobi_result["runtime_sec"] > 0
                    else np.nan
                ),
            }

            detailed_rows.append(row)

    detailed_df = pd.DataFrame(detailed_rows)

    # ------------------------------------------------------------
    # Summary over all instances
    # ------------------------------------------------------------

    summary_overall = pd.DataFrame({
        "metric": [
            "number of instances",
            "D-BH exact success count",
            "D-BH exact success rate",
            "D-BH near success count",
            "D-BH near success rate",
            "D-BH same solution count",
            "D-BH same solution rate",
            "mean absolute gap",
            "median absolute gap",
            "max absolute gap",
            "mean relative gap",
            "max relative gap",
            "mean Hamming distance",
            "max Hamming distance",
            "mean Gurobi runtime",
            "mean D-BH runtime",
            "median Gurobi runtime",
            "median D-BH runtime",
            "mean runtime ratio D-BH/Gurobi",
        ],
        "value": [
            len(detailed_df),
            int(detailed_df["exact_success"].sum()),
            float(detailed_df["exact_success"].mean()),
            int(detailed_df["near_success"].sum()),
            float(detailed_df["near_success"].mean()),
            int(detailed_df["same_solution_string"].sum()),
            float(detailed_df["same_solution_string"].mean()),
            float(detailed_df["abs_gap"].mean()),
            float(detailed_df["abs_gap"].median()),
            float(detailed_df["abs_gap"].max()),
            float(detailed_df["rel_gap"].mean()),
            float(detailed_df["rel_gap"].max()),
            float(detailed_df["hamming_to_gurobi"].mean()),
            float(detailed_df["hamming_to_gurobi"].max()),
            float(detailed_df["gurobi_runtime_sec"].mean()),
            float(detailed_df["dbh_runtime_sec"].mean()),
            float(detailed_df["gurobi_runtime_sec"].median()),
            float(detailed_df["dbh_runtime_sec"].median()),
            float(detailed_df["runtime_ratio_dbh_over_gurobi"].mean()),
        ],
    })

    # ------------------------------------------------------------
    # Summary by problem size
    # ------------------------------------------------------------

    summary_by_n = (
        detailed_df
        .groupby("n", dropna=False)
        .agg(
            instances=("n", "count"),
            exact_successes=("exact_success", "sum"),
            exact_success_rate=("exact_success", "mean"),
            near_successes=("near_success", "sum"),
            near_success_rate=("near_success", "mean"),
            same_solution_count=("same_solution_string", "sum"),
            same_solution_rate=("same_solution_string", "mean"),
            mean_abs_gap=("abs_gap", "mean"),
            max_abs_gap=("abs_gap", "max"),
            mean_rel_gap=("rel_gap", "mean"),
            mean_hamming=("hamming_to_gurobi", "mean"),
            mean_gurobi_runtime_sec=("gurobi_runtime_sec", "mean"),
            mean_dbh_runtime_sec=("dbh_runtime_sec", "mean"),
            mean_runtime_ratio=("runtime_ratio_dbh_over_gurobi", "mean"),
        )
        .reset_index()
    )

    # ------------------------------------------------------------
    # Save CSV files
    # ------------------------------------------------------------

    detailed_path = OUTPUT_DIR / "dbh_vs_gurobi_detailed.csv"
    summary_overall_path = OUTPUT_DIR / "dbh_vs_gurobi_summary_overall.csv"
    summary_by_n_path = OUTPUT_DIR / "dbh_vs_gurobi_summary_by_n.csv"

    detailed_df.to_csv(detailed_path, index=False)
    summary_overall.to_csv(summary_overall_path, index=False)
    summary_by_n.to_csv(summary_by_n_path, index=False)

    print("\n" + "=" * 100)
    print("SUMMARY OVERALL")
    print("=" * 100)
    print(summary_overall.to_string(index=False))

    print("\n" + "=" * 100)
    print("SUMMARY BY n")
    print("=" * 100)
    print(summary_by_n.to_string(index=False))

    print("\n" + "=" * 100)
    print("FILES SAVED")
    print("=" * 100)
    print(detailed_path)
    print(summary_overall_path)
    print(summary_by_n_path)

    return detailed_df, summary_overall, summary_by_n


# ============================================================
# LATEX TABLE GENERATOR
# ============================================================

def make_latex_tables(detailed_df, summary_overall, summary_by_n):
    """
    Create LaTeX tables for the paper.
    """

    # Overall summary table
    latex_overall = summary_overall.to_latex(
        index=False,
        float_format=lambda x: f"{x:.4e}" if abs(x) < 1e-2 or abs(x) > 1e3 else f"{x:.4f}",
        caption="Overall comparison between Gurobi MIQP and D-BH DVQE on hard random QUBO instances.",
        label="tab:dbh_vs_gurobi_overall",
    )

    # By-size summary table
    table_n = summary_by_n.copy()

    table_n = table_n[
        [
            "n",
            "instances",
            "exact_successes",
            "exact_success_rate",
            "near_successes",
            "near_success_rate",
            "mean_abs_gap",
            "max_abs_gap",
            "mean_hamming",
            "mean_gurobi_runtime_sec",
            "mean_dbh_runtime_sec",
        ]
    ]

    latex_by_n = table_n.to_latex(
        index=False,
        float_format=lambda x: f"{x:.4e}" if abs(x) < 1e-2 or abs(x) > 1e3 else f"{x:.4f}",
        caption="D-BH accuracy and runtime relative to Gurobi MIQP by QUBO size.",
        label="tab:dbh_vs_gurobi_by_size",
    )

    # Optional compact detailed table
    compact = detailed_df[
        [
            "n",
            "seed",
            "gurobi_objective",
            "dbh_objective",
            "abs_gap",
            "exact_success",
            "near_success",
            "hamming_to_gurobi",
            "gurobi_runtime_sec",
            "dbh_runtime_sec",
        ]
    ].copy()

    latex_detailed = compact.to_latex(
        index=False,
        float_format=lambda x: f"{x:.4e}" if abs(x) < 1e-2 or abs(x) > 1e3 else f"{x:.4f}",
        caption="Detailed instance-level comparison between Gurobi MIQP and D-BH DVQE.",
        label="tab:dbh_vs_gurobi_detailed",
    )

    with open(OUTPUT_DIR / "table_dbh_vs_gurobi_overall.tex", "w") as f:
        f.write(latex_overall)

    with open(OUTPUT_DIR / "table_dbh_vs_gurobi_by_size.tex", "w") as f:
        f.write(latex_by_n)

    with open(OUTPUT_DIR / "table_dbh_vs_gurobi_detailed.tex", "w") as f:
        f.write(latex_detailed)

    return latex_overall, latex_by_n, latex_detailed


# ============================================================
# OPTIONAL PLOT DATA EXPORT
# ============================================================

def export_plot_data(summary_by_n):
    """
    Save compact data files that can be plotted later in Python, MATLAB, or LaTeX.
    """
    plot_path = OUTPUT_DIR / "plot_data_dbh_vs_gurobi_by_n.csv"

    plot_df = summary_by_n[
        [
            "n",
            "exact_success_rate",
            "near_success_rate",
            "mean_abs_gap",
            "mean_gurobi_runtime_sec",
            "mean_dbh_runtime_sec",
        ]
    ].copy()

    plot_df.to_csv(plot_path, index=False)

    print(f"Plot data saved to: {plot_path}")

    return plot_df




In [64]:
detailed_df, summary_overall, summary_by_n = run_dbh_vs_gurobi_experiment()

latex_overall, latex_by_n, latex_detailed = make_latex_tables(
        detailed_df,
        summary_overall,
        summary_by_n,
    )

plot_df = export_plot_data(summary_by_n)

D-BH DVQE VS GUROBI MIQP ON HARD RANDOM QUBO INSTANCES
Problem sizes: [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
Seeds: [1, 2, 3, 4, 5]
Total QUBO instances: 80
D-BH QPU config: [4, 4, 4, 4, 4]
D-BH settings: {'depth': 1, 'lr': 0.5, 'max_iters': 80, 'rel_tol': 0.0001, 'num_shots': 512, 'final_shots': 5000, 'warm_start_population': 12, 'warm_start_iters': 20, 'warm_start_shots': 512, 'energy_mode': 'cvar', 'cvar_alpha': 0.1, 'verbose': False}

QUBO instance | n= 3 | seed= 1 | qubo_seed=3001
  Gurobi MIQP | obj=-3.28906539 | status=optimal | time= 0.0011s
  D-BH DVQE    | obj=-3.28906539 | gap= 0.000e+00 | exact=True | near=True | time= 0.4582s

QUBO instance | n= 3 | seed= 2 | qubo_seed=3002
  Gurobi MIQP | obj=-1.35967972 | status=optimal | time= 0.0010s
  D-BH DVQE    | obj=-1.35967972 | gap= 0.000e+00 | exact=True | near=True | time= 0.4436s

QUBO instance | n= 3 | seed= 3 | qubo_seed=3003
  Gurobi MIQP | obj=-0.26852856 | status=optimal | time= 0.0010s
  D-BH DVQE    